# Order Status Agent

This agent handles customer inquiries about:
- Order tracking
- Shipping status
- Delivery estimates
- Order history

## Setup

In [ ]:
# Import required libraries
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

# Load configuration
from config_loader import load_config, get_model_name
config = load_config()
model_name = get_model_name(config)

## Define Agent State

In [ ]:
class AgentState(TypedDict):
    """State for order status agent"""
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Create Order Status Agent

In [ ]:
# System prompt for order status agent
ORDER_STATUS_PROMPT = """You are a helpful customer service agent specializing in ORDER TRACKING and SHIPPING INQUIRIES.

Your responsibilities:
- Help customers track their orders
- Provide shipping status updates
- Estimate delivery times
- Explain order statuses (processing, shipped, delivered, etc.)
- Locate order details using order numbers

For demonstration purposes:
- Order numbers are in format: ORD-XXXXX
- Typical delivery time is 3-5 business days
- Orders are shipped via standard carrier

Be friendly, professional, and provide clear information about order status.
If you don't have specific order details, provide general guidance on how to track orders.
"""

def create_order_status_agent():
    """Create the order status agent"""
    
    # Initialize LLM
    llm = ChatOpenAI(model=model_name, temperature=0.7)
    
    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", ORDER_STATUS_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    # Create agent function
    def agent_node(state: AgentState):
        messages = state["messages"]
        response = llm.invoke(prompt.format_messages(messages=messages))
        return {"messages": [response]}
    
    # Create graph
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)
    
    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    
    return app

# Create the agent
order_status_agent = create_order_status_agent()
print("✅ Order Status Agent created successfully!")

## Test the Agent

In [ ]:
# Test conversation
from uuid import uuid4

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Test query 1
print("User: Where is my order?\n")
result = order_status_agent.invoke(
    {"messages": [HumanMessage(content="Where is my order?")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)

In [ ]:
# Test query 2 (with context)
print("User: My order number is ORD-12345\n")
result = order_status_agent.invoke(
    {"messages": [HumanMessage(content="My order number is ORD-12345")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)

In [ ]:
# Test query 3 (follow-up)
print("User: When will it arrive?\n")
result = order_status_agent.invoke(
    {"messages": [HumanMessage(content="When will it arrive?")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)
print("\n✅ Agent maintained context across 3 turns!")